In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
import torch.nn as nn

class SubtractOne(nn.Module):
  def forward(self, img):
    return img-1

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28:
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor:
    transforms.ToTensor(),
    SubtractOne() ,
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])

])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, drop_last=True)


# Write your code here
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")

import matplotlib.pyplot as plt
import numpy as np

# Define mean & std for denormalization (EfficientNet Preprocessing)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [270,233,110,89,15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image & label

    # Convert tensor to numpy for visualization
    # always when you retreive one image you get a dimension (c,h,w)
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)

    # Denormalize the image
    img_np = std * img_np + mean # denormalized is by inverse the process
    img_np = np.clip(img_np, 0, 1)
# why it deasn't flat ax?? because it is one dimension so only one row
# if it is n't one dimemsion so you need use ax[][] as grid (two dimnsional array), or you can normalize
    # Show image
    axes[i].imshow(img_np)
    axes[i].set_title(f'Class: {label}')
    axes[i].axis('off')

plt.show()



In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

device = "cuda" if torch.cuda.is_available() else "cpu"

# Write your code here
model = efficientnet_v2_s(pretrained=True).to(device)
model.requires_grad_(False)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)  # 1 = 2 classes
model.classifier.requires_grad_(True)





In [ ]:
# model

In [ ]:
# Write your code here
from tqdm import tqdm    # Shows progress bar

# def remap_mask(mask):
#     # Remaps a mask's pixel values to a consecutive range starting at 0 (Use if Multiclass Segmentation)
#     mask = mask.long()
#     unique_values = torch.unique(mask)
#     # Creates a blank canvas (filled with zeros)
#     remapped_mask = torch.zeros_like(mask)
#                             # enumerate returnes counter i start frim 0 which is here is new_val
#     for new_val, old_val in enumerate(sorted(unique_values.tolist())):
#         remapped_mask[mask == old_val] = new_val

#     return remapped_mask
#     for images, labels in tqdm(train_loader):
#       lables = remap_mask(labels)
#     for images, labels in tqdm(test_loader):
#       lables = remap_mask(labels)


# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        # explained in day 1 lab 1
        # Track accuracy
        # see day 1 lab 1 and tell me why we did't do there softmax like here answer is se the last line and model structure
        # there is no cross entropy loss function to internally doras softmax! so explicitly write it plz
        # Get class with highest probability, argmax gives the maximum value index, multople classes indexes are clases
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

    # 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

In [ ]:
# Write your code here
import torch.optim as optim


# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")



In [ ]:
# Write your code here
